In [1]:
import numpy as np
import pandas as pd
import Utils

In [2]:
LANGUAGE = Utils.LANGUAGE_CZ

data_path = path = Utils.get_SoD_dataset_path(LANGUAGE)
df = pd.read_spss(data_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3880 entries, 0 to 3879
Columns: 170 entries, id to lca_DK_n_8
dtypes: category(163), float64(7)
memory usage: 864.0 KB


In [11]:
def process_SoD_employment(row):
    # Column names for employment statuses contain 'EMPLOYMENT_' and number from range from 1 to 10.
    employment_cols = [col for col in row.index if 'EMPLOYMENT_' in col]

    # Filter out 'Ne' (No) responses and collect the others.
    employment_statuses = [row[col] for col in employment_cols if row.get(col) != 'Ne']

    return ', '.join(employment_statuses).lower()

def process_SoD_income(row):
    income_raw = row.get('INCOMEP', '')

    if not income_raw or 'Nevím' in income_raw:
        return None

    return income_raw[0].lower() + income_raw[1:]

def process_SoD_town_size(row):
    income_raw = row.get('VMB', '')

    if 'Méně než 1.000' in income_raw:
        income_raw+=' obyvatel'

    return income_raw[0].lower() + income_raw[1:]


def process_SoD_response(row):
    # --- 1. Basic Information ---
    # Renames Czech keys to English and performs initial data cleaning.
    processed_data = {
        'gender': row.get('GENDER'),
        'age': int(row.get('AGE1', 0)),
        'education_level': row.get('EDU', '').lower(),
        'region': row.get('KRAJ'),
        'district': row.get('OKRES'),
        'town_size': process_SoD_town_size(row),
        'employment_status': process_SoD_employment(row),
        'income_range': process_SoD_income(row),
    }

    # --- 2. Additional Survey Questions ---
    # Merges the dictionary of additional questions into the main one.
    additional_data = {
        'living_standard': row.get('Q19'),
        'interest_in_politics': row.get('Q20'),
        'opinion_on_eu': row.get('Q18'),
        'opinion_on_nato': row.get('Q17'),
        'covid_vaccinated': row.get('Q23'),
    }
    processed_data.update(additional_data)


    return processed_data

data = [process_SoD_response(df.loc[i]) for i in df.index]
data = pd.DataFrame(data, index=df.index)

In [12]:
# --- Helper Functions for Text Formatting ---
def _format_gender_a(word, gender):
    return word if gender == 'Muž' else word+'a'

def _format_gender_ya(word, gender):
    return word+'ý' if gender == 'Muž' else word+'á'

def _format_negation_ne(word, capitalize=False):
    if capitalize:
        return 'Ne'+word.lower()
    else:
        return 'ne'+word.lower()

def _format_region_for_sentence(region_name):
    """
    Applies Czech grammar rules to correctly decline a region name for use
    in a sentence like "Žiji v [region name]".

    For example: 'Plzeňský kraj' -> 'Plzeňském kraji'
    """
    # Handles special case for Prague 'Hlavní město Praha'
    if 'Praha' in region_name:
        return 'Praze'

    # General grammar rules for other regions
    declined_name = region_name.replace('raj', 'raji')  # Covers Kraj and kraj
    if 'ký' in declined_name:
        declined_name = declined_name.replace('ký', 'kém')

    return declined_name

def _format_opinion_statement(gender, opinion, topic_string):
    """
    Creates a full opinion sentence with correct gendered adjectives.
    Example: (gender='Muž', opinion='Rozhodně ano', topic='EU')
             -> "Jsem rozhodně přesvědčený, že je Česká republika členem EU."

    This helper removes code duplication for the EU and NATO questions.
    """
    if opinion == 'Nevím':
        return "" # Return an empty string if there is no opinion

    return f"Jsem {_format_gender_ya(opinion[:-3],gender).lower()}, že je Česká republika členským státem {topic_string}."
# --- Main Function to Create the Description ---

def create_respondent_description(respondent):
    """
    Generates a descriptive Czech paragraph about a survey respondent
    by combining their answers into grammatically correct sentences.

    Args:
        respondent (dict): A dictionary containing the processed data for one person,
                           with English keys (e.g., 'gender', 'region').

    Returns:
        str: A multi-sentence description of the respondent in Czech.
    """
    # --- 1. Build the description sentence by sentence ---
    # Using a list of sentences is cleaner than repeated string concatenation.
    description_parts = []

    # --- Basic Demographics ---
    description_parts.append(
        f"Jsem {respondent['gender'].lower()},"
        f"je mi {respondent['age']} let, "
        f"mé vzdělání je {respondent['education_level']}."
    )

    # --- Location ---
    # This now uses the helper function for complex Czech grammar.
    region_in_sentence = _format_region_for_sentence(respondent['region'])
    description_parts.append(
        f"Žiji v {region_in_sentence},"
        f" v okresu {respondent['district']} a "
        f"obci o velikosti {respondent['town_size']}."
    )

    # --- Socioeconomic Status ---
    employment_text = f"Z hlediska zaměstnání jsem {respondent['employment_status']}"
    if respondent['income_range']:
        employment_text += f" a příjem naší domácnosti je {respondent['income_range']}"
    employment_text += "." # End the sentence if there is no income data.
    description_parts.append(employment_text)

    # --- Opinions and Beliefs ---
    # Gender suffix for words like "očkován/očkována"
    gender_suffix_a = '' if respondent['gender'] == 'Muž' else 'a'

    # EU and NATO opinions now use the dedicated helper function
    description_parts.append(_format_opinion_statement(respondent['gender'], respondent['opinion_on_eu'], "EU"))
    description_parts.append(_format_opinion_statement(respondent['gender'], respondent['opinion_on_nato'], "NATO"))

    # Living Standard
    living_standard = respondent['living_standard']
    if living_standard != 'Nevím':
        # This logic determines if the verb should be positive ('Mám') or negative ('Nemám')
        verb = _format_negation_ne('Mám',True)
        description_parts.append(f"{verb} {living_standard.lower()} životní úroveň.")

    # Interest in Politics
    interest = respondent['interest_in_politics']
    if interest != 'Nevím':
        description_parts.append(f"{interest} o politiku.")

    # COVID Vaccination Status
    vacc_status = respondent['covid_vaccinated']
    sentence = f"očkován{gender_suffix_a} proti covidu."
    if vacc_status == 'Ano':
        description_parts.append(f"Jsem {sentence}")
    elif vacc_status == 'Ne':
        description_parts.append(f"Nejsem {sentence}")

    # --- 2. Combine all parts into a final paragraph ---
    # Filter out any empty strings that may have been returned by helpers (e.g., for 'Nevím' answers)
    # and join the parts with a space.
    full_description = " ".join(part for part in description_parts if part)

    return full_description

In [13]:
data

,gender,age,education_level,region,district,town_size,employment_status,income_range,living_standard,interest_in_politics,opinion_on_eu,opinion_on_nato,covid_vaccinated
0,Muž,28,vysokoškolské vzdělání,Středočeský kraj,Nymburk,méně než 1.000 obyvatel,zaměstnanec na plný úvazek,30.001 - 40.000 Kč,Spíše dobrou,Velmi se zajímám,Spíše spokojený/á,Rozhodně spokojený/á,Ano
1,Muž,27,vysokoškolské vzdělání,Plzeňský kraj,Plzeň-město,více než 100.000 obyvatel,zaměstnanec na plný úvazek,20.001 - 25.000 Kč,Spíše dobrou,Spíše se zajímám,Spíše spokojený/á,Spíše spokojený/á,Ano
2,Žena,33,základní + středoškolské vzdělání bez maturity,Královéhradecký kraj,Jičín,méně než 1.000 obyvatel,zaměstnanec na plný úvazek,25.001 - 30.000 Kč,"Ani dobrou, ani špatnou",Spíše se zajímám,Spíše spokojený/á,Rozhodně spokojený/á,Ano
3,Žena,27,základní + středoškolské vzdělání bez maturity,Plzeňský kraj,Rokycany,20.001 - 100.000 obyvatel,zaměstnanec na plný úvazek,20.001 - 25.000 Kč,Spíše dobrou,Spíše se zajímám,Rozhodně spokojený/á,Rozhodně spokojený/á,Ano
4,Muž,21,středoškolské vzdělání s maturitou,Ústecký kraj,Chomutov,20.001 - 100.000 obyvatel,zaměstnanec na plný úvazek,30.001 - 40.000 Kč,Velmi dobrou,Vůbec se nezajímám,Rozhodně spokojený/á,Rozhodně spokojený/á,Ano
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3875,Žena,39,středoškolské vzdělání s maturitou,Plzeňský kraj,Plzeň-město,více než 100.000 obyvatel,zaměstnanec na plný úvazek,40.001 - 60.000 Kč,Spíše dobrou,Spíše se nezajímám,Spíše spokojený/á,Spíše spokojený/á,Ne
3876,Žena,30,vysokoškolské vzdělání,Hlavní město Praha,Praha,více než 100.000 obyvatel,zaměstnanec na částečný úvazek,None,Spíše dobrou,Spíše se zajímám,Rozhodně spokojený/á,Rozhodně spokojený/á,Ano
3877,Žena,47,vysokoškolské vzdělání,Moravskoslezský kraj,Karviná,5.001 - 20.000 obyvatel,zaměstnanec na plný úvazek,40.001 - 60.000 Kč,Spíše dobrou,Vůbec se nezajímám,Spíše spokojený/á,Rozhodně spokojený/á,Ano
3878,Žena,23,středoškolské vzdělání s maturitou,Hlavní město Praha,Praha,20.001 - 100.000 obyvatel,zaměstnanec na plný úvazek,20.001 - 25.000 Kč,Spíše špatnou,Nevím,Nevím,Nevím,Nechci uvést


In [6]:
create_respondent_description(data.iloc[0])

'Jsem muž,je mi 28 let, mé vzdělání je vysokoškolské vzdělání. Žiji v Středočeském kraji, v okresu Nymburk a obci o velikosti Méně než 1.000. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek a příjem naší domácnosti je 30.001 - 40.000 Kč. Jsem spíše spokojený, že je Česká republika členským státem EU. Jsem rozhodně spokojený, že je Česká republika členským státem NATO. Nemám spíše dobrou životní úroveň. Velmi se zajímám o politiku. Jsem očkován proti covidu.'

In [9]:
czech_prompt_start = """Po značce [INSERT] doplň, zda respondent volil ve volbách do poslanecké sněmovny 2021 a pokud ano, pro jakou stranu hlasoval.
Možné strany, kterým nejspíše dal hlas, vypiš s pravděpodobností že pro danou stranu hlasoval. Vypiš kolik stran je potřeba.
Případně můžeš vypsat i "jiná strana", což představuje hlas pro málo populární stranu. Součet pravděpodobností musí být 1.0.

Formát výstupu je:
*[volil, proba a], [nevolil, proba b]; [PARTY1, proba 1], [PARTY2, proba 2],... , [PARTYN, proba n]*
 kde PARTY1, PARTY2,... jsou jména stram, proba a, proba b, proba 1,... je třeba nahradit desetinnými čísly (pravděpodobnost). Za druhý znak "*" už nic nevypisuj.

 Zachovej správné oddělovače (';' mezi sekcí volil/nevolil a sekcí stran, všude jinde ','). Volil/nevolil je první, pak následují strany.
"""

czech_prompt_end = " Ve volbách do poslanecké sněmovny 2021 jsem [INSERT]"

In [10]:
ex_prompt = czech_prompt_start + create_respondent_description(data.iloc[0]) + czech_prompt_end
ex_prompt

'Po značce [INSERT] doplň, zda respondent volil ve volbách do poslanecké sněmovny 2021 a pokud ano, pro jakou stranu hlasoval.\nMožné strany, kterým nejspíše dal hlas, vypiš s pravděpodobností že pro danou stranu hlasoval. Vypiš kolik stran je potřeba.\nPřípadně můžeš vypsat i "jiná strana", což představuje hlas pro málo populární stranu. Součet pravděpodobností musí být 1.0.\n\nFormát výstupu je:\n*[volil, proba a], [nevolil, proba b]; [PARTY1, proba 1], [PARTY2, proba 2],... , [PARTYN, proba n]*\n kde PARTY1, PARTY2,... jsou jména stram, proba a, proba b, proba 1,... je třeba nahradit desetinnými čísly (pravděpodobnost). Za druhý znak "*" už nic nevypisuj.\n\n Zachovej správné oddělovače (\';\' mezi sekcí volil/nevolil a sekcí stran, všude jinde \',\'). Volil/nevolil je první, pak následují strany.\nJsem muž,je mi 28 let, mé vzdělání je vysokoškolské vzdělání. Žiji v Středočeském kraji, v okresu Nymburk a obci o velikosti Méně než 1.000. Z hlediska zaměstnání jsem zaměstnanec na plný

In [ ]:
#TODO: GPT 4 nano
# pydock structured data
# poslat vysledky
# ciel : structured output works